# Mesh Tutorial 6: Integration with ML Workflows

This tutorial covers advanced topics for using PhysicsNeMo-Mesh in ML pipelines:

1. **Performance Comparison**: PhysicsNeMo-Mesh vs PyVista/VTK
2. **GPU Acceleration Benefits**: When and how much speedup to expect
3. **Batching Meshes**: Padding for torch.compile compatibility
4. **Feature Extraction**: Preparing mesh data for ML models
5. **Boundary Condition Handling**: Storing BC metadata in TensorDict
6. **End-to-End Workflow**: Complete CAE preprocessing example

---

## Why Replace PyVista/VTK in ML Pipelines?

Traditional mesh libraries like PyVista and VTK are CPU-bound:

- **CPU-GPU transfers**: Data must be copied to GPU for each training step
- **GIL bottleneck**: Python's Global Interpreter Lock limits parallelism
- **No autograd**: Cannot backpropagate through mesh operations

PhysicsNeMo-Mesh solves these by being:
- **GPU-native**: All operations run on CUDA
- **Differentiable**: Integrates with PyTorch autograd
- **TensorDict-based**: Efficient batching and device management

In [ ]:
import torch
import time
import pyvista as pv
from tensordict import TensorDict

from physicsnemo.mesh import Mesh
from physicsnemo.mesh.io import from_pyvista
from physicsnemo.mesh.primitives.surfaces import sphere_icosahedral

## Section 1: Performance Comparison

Let's compare PhysicsNeMo-Mesh to PyVista for common operations.

In [ ]:
def benchmark(name, func, n_runs=5, warmup=2):
    """Benchmark a function with warmup runs."""
    # Warmup
    for _ in range(warmup):
        func()
    
    # Timed runs
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        result = func()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    
    mean_time = sum(times) / len(times)
    print(f"{name}: {mean_time*1000:.2f} ms")
    return mean_time, result

### Normal Computation

In [ ]:
# Create test meshes of increasing size
mesh_sizes = [2, 3, 4]  # subdivision levels

print("Normal Computation Benchmark")
print("=" * 50)

for subdiv in mesh_sizes:
    # Create meshes
    pnm_mesh = sphere_icosahedral.load(subdivisions=subdiv)
    pv_mesh = pv.Sphere(theta_resolution=2**(subdiv+2), phi_resolution=2**(subdiv+2))
    
    print(f"\nMesh size: {pnm_mesh.n_points} points, {pnm_mesh.n_cells} cells")
    
    # PyVista (CPU)
    def pyvista_normals():
        return pv_mesh.compute_normals(cell_normals=True, point_normals=False)
    
    # PhysicsNeMo-Mesh (CPU)
    def pnm_cpu_normals():
        return pnm_mesh.cell_normals
    
    pv_time, _ = benchmark("  PyVista (CPU)", pyvista_normals)
    pnm_cpu_time, _ = benchmark("  PhysicsNeMo (CPU)", pnm_cpu_normals)
    
    # PhysicsNeMo-Mesh (GPU)
    if torch.cuda.is_available():
        pnm_gpu_mesh = pnm_mesh.to("cuda")
        
        def pnm_gpu_normals():
            return pnm_gpu_mesh.cell_normals
        
        pnm_gpu_time, _ = benchmark("  PhysicsNeMo (GPU)", pnm_gpu_normals)
        print(f"  Speedup (GPU vs PyVista): {pv_time/pnm_gpu_time:.1f}x")

### Curvature Computation

In [ ]:
print("Curvature Computation Benchmark")
print("=" * 50)

for subdiv in [3, 4]:
    # Create meshes
    pnm_mesh = sphere_icosahedral.load(subdivisions=subdiv)
    pv_mesh = pnm_mesh.to_pyvista() if hasattr(pnm_mesh, 'to_pyvista') else None
    
    print(f"\nMesh size: {pnm_mesh.n_points} points")
    
    # PhysicsNeMo-Mesh (CPU)
    def pnm_cpu_curvature():
        return pnm_mesh.gaussian_curvature_vertices
    
    pnm_cpu_time, _ = benchmark("  PhysicsNeMo (CPU)", pnm_cpu_curvature)
    
    # PhysicsNeMo-Mesh (GPU)
    if torch.cuda.is_available():
        pnm_gpu_mesh = pnm_mesh.to("cuda")
        
        def pnm_gpu_curvature():
            return pnm_gpu_mesh.gaussian_curvature_vertices
        
        pnm_gpu_time, _ = benchmark("  PhysicsNeMo (GPU)", pnm_gpu_curvature)
        print(f"  GPU speedup: {pnm_cpu_time/pnm_gpu_time:.1f}x")

## Section 2: Batching Meshes for Training

When training with meshes of varying sizes, you need to handle dynamic shapes.
PhysicsNeMo-Mesh provides padding utilities for this.

In [ ]:
# Create meshes of different sizes
meshes = [
    sphere_icosahedral.load(subdivisions=1),
    sphere_icosahedral.load(subdivisions=2),
    sphere_icosahedral.load(subdivisions=3),
]

print("Original mesh sizes:")
for i, m in enumerate(meshes):
    print(f"  Mesh {i}: {m.n_points} points, {m.n_cells} cells")

### Fixed-Size Padding

In [ ]:
# Pad all meshes to fixed sizes
max_points = max(m.n_points for m in meshes)
max_cells = max(m.n_cells for m in meshes)

padded_meshes = [m.pad(target_n_points=max_points, target_n_cells=max_cells) for m in meshes]

print(f"\nPadded to {max_points} points, {max_cells} cells:")
for i, m in enumerate(padded_meshes):
    print(f"  Mesh {i}: {m.n_points} points, {m.n_cells} cells")

### Power-Based Padding (for torch.compile)

For `torch.compile` with `dynamic=False`, pad to the next power of a base.
This limits the number of compiled kernel variants.

In [ ]:
# Pad to next power of 1.5 (balances memory vs. compile cache hits)
power_padded = [m.pad_to_next_power(power=1.5) for m in meshes]

print("Power-padded sizes:")
for i, (orig, padded) in enumerate(zip(meshes, power_padded)):
    print(f"  Mesh {i}: {orig.n_points} -> {padded.n_points} points, "
          f"{orig.n_cells} -> {padded.n_cells} cells")

## Section 3: Feature Extraction for ML

Prepare geometric and physical features for model input.

In [ ]:
def extract_features(mesh):
    """Extract features for an ML model."""
    features = TensorDict({}, batch_size=[mesh.n_points])
    
    # Geometric features
    features["position"] = mesh.points
    
    if mesh.codimension == 1:  # Surface mesh
        features["normal"] = mesh.point_normals
        features["gaussian_curvature"] = mesh.gaussian_curvature_vertices.unsqueeze(-1)
        features["mean_curvature"] = mesh.mean_curvature_vertices.unsqueeze(-1)
    
    return features

In [ ]:
# Example: extract features from bunny mesh
bunny = torch.load("assets/bunny.pt", weights_only=False).subdivide(2, "loop")

features = extract_features(bunny)

print("Extracted features:")
for key in features.keys():
    print(f"  {key}: {features[key].shape}")

In [ ]:
# Concatenate into feature matrix
feature_matrix = torch.cat([
    features["position"],
    features["normal"],
    features["gaussian_curvature"],
    features["mean_curvature"],
], dim=-1)

print(f"Feature matrix: {feature_matrix.shape}")
print(f"  (n_points, n_features)")

## Section 4: Boundary Condition Handling

A key advantage of PhysicsNeMo-Mesh is the ability to store rich metadata,
including boundary condition information.

In [ ]:
# Example: CFD mesh with boundary conditions
mesh = sphere_icosahedral.load(subdivisions=3)

# Define BC types
BC_INTERIOR = 0
BC_INLET = 1
BC_OUTLET = 2
BC_WALL = 3

# Assign BC types based on position (example)
x = mesh.points[:, 0]
bc_type = torch.full((mesh.n_points,), BC_INTERIOR, dtype=torch.long)
bc_type[x < -0.8] = BC_INLET
bc_type[x > 0.8] = BC_OUTLET
bc_type[(x >= -0.8) & (x <= 0.8) & (mesh.points[:, 2] < 0)] = BC_WALL

mesh.point_data["bc_type"] = bc_type

print("Boundary condition counts:")
for name, val in [("Interior", 0), ("Inlet", 1), ("Outlet", 2), ("Wall", 3)]:
    count = (bc_type == val).sum().item()
    print(f"  {name}: {count}")

In [ ]:
# Visualize BC types
mesh.draw(point_scalars="bc_type", cmap="Set1", show_edges=False)

In [ ]:
# Store BC values in nested TensorDict
mesh.point_data["bc_values"] = TensorDict({
    "velocity": torch.zeros(mesh.n_points, 3),
    "pressure": torch.full((mesh.n_points,), float('nan')),
}, batch_size=[mesh.n_points])

# Set inlet velocity (1 m/s in x-direction)
inlet_mask = mesh.point_data["bc_type"] == BC_INLET
mesh.point_data["bc_values", "velocity"][inlet_mask] = torch.tensor([1.0, 0.0, 0.0])

# Set outlet pressure (0 Pa gauge)
outlet_mask = mesh.point_data["bc_type"] == BC_OUTLET
mesh.point_data["bc_values", "pressure"][outlet_mask] = 0.0

print(f"Mesh with BCs: {mesh}")

## Section 5: End-to-End CAE Workflow

Complete example: load mesh, extract features, prepare for GNN training.

In [ ]:
def prepare_mesh_for_training(mesh, device="cpu"):
    """
    Prepare a mesh for GNN training.
    
    Returns:
        node_features: (n_nodes, n_features)
        edge_index: (2, n_edges)
        edge_features: (n_edges, n_edge_features)
    """
    mesh = mesh.to(device)
    
    ### Node features: position + geometric features
    node_features = [mesh.points]
    
    if mesh.codimension == 1:
        node_features.append(mesh.point_normals)
        node_features.append(mesh.gaussian_curvature_vertices.unsqueeze(-1))
        node_features.append(mesh.mean_curvature_vertices.unsqueeze(-1))
    
    node_features = torch.cat(node_features, dim=-1)
    
    ### Edge index from mesh adjacency
    adj = mesh.get_point_to_points_adjacency()
    neighbor_counts = adj.offsets[1:] - adj.offsets[:-1]
    source = torch.repeat_interleave(
        torch.arange(mesh.n_points, device=device), 
        neighbor_counts
    )
    target = adj.indices
    edge_index = torch.stack([source, target], dim=0)
    
    ### Edge features: relative position, distance
    edge_vectors = mesh.points[target] - mesh.points[source]
    edge_lengths = edge_vectors.norm(dim=-1, keepdim=True)
    edge_features = torch.cat([edge_vectors, edge_lengths], dim=-1)
    
    return {
        "node_features": node_features,
        "edge_index": edge_index,
        "edge_features": edge_features,
        "n_nodes": mesh.n_points,
        "n_edges": edge_index.shape[1],
    }

In [ ]:
# Example usage
bunny = torch.load("assets/bunny.pt", weights_only=False).subdivide(2, "loop")

device = "cuda" if torch.cuda.is_available() else "cpu"
graph_data = prepare_mesh_for_training(bunny, device=device)

print("GNN-ready data:")
for key, value in graph_data.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape} on {value.device}")
    else:
        print(f"  {key}: {value}")

## Section 6: torch.compile Compatibility

Most PhysicsNeMo-Mesh operations are compatible with `torch.compile`.

In [ ]:
if torch.cuda.is_available():
    @torch.compile
    def compute_features_compiled(points, cells):
        """Compiled feature computation."""
        mesh = Mesh(points=points, cells=cells)
        normals = mesh.cell_normals
        areas = mesh.cell_areas
        return normals, areas
    
    # Create test mesh on GPU
    mesh = sphere_icosahedral.load(subdivisions=3).to("cuda")
    
    print("Testing torch.compile...")
    normals, areas = compute_features_compiled(mesh.points, mesh.cells)
    print(f"  Normals: {normals.shape}")
    print(f"  Areas: {areas.shape}")
    print("  Success!")
else:
    print("CUDA not available - skipping torch.compile demo")

## Summary

In this tutorial, you learned about ML integration:

1. **Performance**: GPU acceleration provides significant speedups
2. **Batching**: `pad()` and `pad_to_next_power()` for dynamic shapes
3. **Feature Extraction**: Geometric features for model input
4. **Boundary Conditions**: Nested TensorDict for rich metadata
5. **End-to-End**: Complete workflow from mesh to GNN-ready data
6. **torch.compile**: Most operations are compilation-compatible

---

### Conclusion

You've completed the PhysicsNeMo-Mesh tutorial series! You now know how to:

- Create, load, and manipulate meshes
- Perform geometric transformations and subdivision
- Compute gradients, divergence, curl, and curvature
- Query neighbors and perform spatial searches
- Validate and repair meshes
- Integrate meshes into ML training pipelines

For more details, see the [physicsnemo.mesh README](../../../physicsnemo/mesh/README.md).